# Rayhana Basil Disease Detection with YOLOv8

Rayhana is a basil-only smart care app. This notebook trains the first object detection baseline for basil leaf conditions: `fusarium`, `healthy`, and `powdery`.

The GitHub repository is code-only. The Roboflow dataset is downloaded inside Colab, staged as raw data, and then prepared into a fresh train/valid/test split before training.

## 1. Environment Setup

Use a Colab GPU runtime for training. These first cells clone the code-only repository and install the Python packages used by the notebook.

In [ ]:
import subprocess

gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_check.returncode == 0:
    print(gpu_check.stdout.splitlines()[0])
else:
    print('No GPU detected. In Colab, switch Runtime > Change runtime type > GPU before training.')

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/Mohamed-515/rayhana-ai.git'
PROJECT_DIR = Path('/content/rayhana-ai')

if not (PROJECT_DIR / 'src').exists():
    !git clone -q {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)

In [ ]:
!pip -q install -r requirements.txt roboflow

In [ ]:
from collections import Counter
from pathlib import Path
import json
import os
import random
import shutil

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from PIL import Image, ImageDraw, ImageFont, UnidentifiedImageError
from ultralytics import YOLO

In [ ]:
DATASET_DIR = PROJECT_DIR / 'data/processed/basil_yolo_detection'
DATA_YAML = DATASET_DIR / 'data.yaml'
RAW_ROBOFLOW_DIR = PROJECT_DIR / 'data/raw/basil_roboflow_yolov8'

REPORTS_DIR = PROJECT_DIR / 'reports'
METRICS_DIR = REPORTS_DIR / 'metrics'
FIGURES_DIR = REPORTS_DIR / 'figures'
YOLO_PROJECT = str(REPORTS_DIR / 'yolo')
RUN_NAME = 'basil_detection'

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Raw dataset target:', RAW_ROBOFLOW_DIR)
print('Processed dataset target:', DATASET_DIR)

## 2. Download Dataset from Roboflow

Download the Basil YOLOv8 export inside Colab. Use either the pasted-code cell or the API-key cell, then continue to the staging cell.

In [ ]:
# Option A: paste Roboflow's generated YOLOv8 download code below.
# It should end with something like: dataset = version.download("yolov8")
# Leave this cell unchanged if you prefer the API-key cell.

ROBOFLOW_DATASET_DIR = None

# Paste Roboflow code under this line.
# from roboflow import Roboflow
# rf = Roboflow(api_key="PASTE_API_KEY")
# project = rf.workspace("WORKSPACE").project("PROJECT")
# version = project.version(1)
# dataset = version.download("yolov8")

if 'dataset' in globals() and hasattr(dataset, 'location'):
    ROBOFLOW_DATASET_DIR = Path(dataset.location)
    print('Roboflow dataset:', ROBOFLOW_DATASET_DIR)
else:
    print('Paste Roboflow code here, or use the API-key cell below.')

In [ ]:
# Option B: fill these values, or store the key in Colab Secrets as ROBOFLOW_API_KEY.
ROBOFLOW_API_KEY = ''
ROBOFLOW_WORKSPACE = ''
ROBOFLOW_PROJECT = ''
ROBOFLOW_VERSION = ''

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = ROBOFLOW_API_KEY or (userdata.get('ROBOFLOW_API_KEY') or '')
except Exception:
    pass

if ROBOFLOW_API_KEY and ROBOFLOW_WORKSPACE and ROBOFLOW_PROJECT and ROBOFLOW_VERSION:
    from roboflow import Roboflow

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    dataset = project.version(int(ROBOFLOW_VERSION)).download('yolov8')
    ROBOFLOW_DATASET_DIR = Path(dataset.location)
    print('Roboflow dataset:', ROBOFLOW_DATASET_DIR)
else:
    print('Skipped API-key download. Fill the four values above if you are not using pasted Roboflow code.')

In [ ]:
if ROBOFLOW_DATASET_DIR is not None:
    ROBOFLOW_DATASET_DIR = Path(ROBOFLOW_DATASET_DIR)

if ROBOFLOW_DATASET_DIR is None:
    candidates = []
    for data_yaml in Path('/content').rglob('data.yaml'):
        candidate = data_yaml.parent
        has_yolo_split = any((candidate / split / 'images').exists() for split in ['train', 'valid', 'test'])
        if has_yolo_split:
            candidates.append(candidate)
    candidates = sorted(candidates, key=lambda path: path.stat().st_mtime, reverse=True)
    ROBOFLOW_DATASET_DIR = candidates[0] if candidates else None

assert ROBOFLOW_DATASET_DIR is not None, 'Download the Roboflow YOLOv8 dataset first.'
assert ROBOFLOW_DATASET_DIR.exists(), f'Missing Roboflow folder: {ROBOFLOW_DATASET_DIR}'

print('Using Roboflow folder:', ROBOFLOW_DATASET_DIR)

## 3. Prepare Train/Valid/Test Split

Roboflow exports can arrive with one split or several splits. This staging step collects all available YOLO image/label pairs into the raw folder expected by the project script. The script then creates Rayhana's reproducible 70/20/10 processed split.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}


def roboflow_split_dirs(root):
    split_dirs = []
    for split in ['train', 'valid', 'test']:
        image_dir = root / split / 'images'
        label_dir = root / split / 'labels'
        if image_dir.exists() and label_dir.exists():
            split_dirs.append((split, image_dir, label_dir))

    image_dir = root / 'images'
    label_dir = root / 'labels'
    if image_dir.exists() and label_dir.exists():
        split_dirs.append(('root', image_dir, label_dir))

    return split_dirs


def stage_roboflow_for_rayhana(source_root, raw_root):
    split_dirs = roboflow_split_dirs(source_root)
    assert split_dirs, f'No YOLO image/label folders found under {source_root}'

    if raw_root.exists():
        shutil.rmtree(raw_root)

    target_images = raw_root / 'train' / 'images'
    target_labels = raw_root / 'train' / 'labels'
    target_images.mkdir(parents=True, exist_ok=True)
    target_labels.mkdir(parents=True, exist_ok=True)

    staged = []
    missing_labels = []

    for split, image_dir, label_dir in split_dirs:
        for image_path in sorted(image_dir.iterdir()):
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            label_path = label_dir / f'{image_path.stem}.txt'
            if not label_path.exists():
                missing_labels.append((split, image_path.name))
                continue

            target_stem = image_path.stem
            target_image_path = target_images / f'{target_stem}{image_path.suffix}'
            target_label_path = target_labels / f'{target_stem}.txt'

            if target_image_path.exists() or target_label_path.exists():
                target_stem = f'{split}_{image_path.stem}'
                target_image_path = target_images / f'{target_stem}{image_path.suffix}'
                target_label_path = target_labels / f'{target_stem}.txt'

            shutil.copy2(image_path, target_image_path)
            shutil.copy2(label_path, target_label_path)
            staged.append({'source_split': split, 'image': target_image_path.name})

    return staged, missing_labels


staged_pairs, missing_raw_labels = stage_roboflow_for_rayhana(ROBOFLOW_DATASET_DIR, RAW_ROBOFLOW_DIR)

print(f'Staged image/label pairs: {len(staged_pairs)}')
print(f'Missing labels skipped: {len(missing_raw_labels)}')
assert staged_pairs, 'No labeled images were staged.'

In [ ]:
!python -m src.data.prepare_basil_yolo_split

assert DATA_YAML.exists(), f'Missing processed data.yaml: {DATA_YAML}'
print('Processed dataset ready:', DATASET_DIR)

In [ ]:
split_report_path = PROJECT_DIR / 'reports/basil_yolo_split_report.json'
split_report = json.loads(split_report_path.read_text(encoding='utf-8'))
split_report

## 4. Dataset Exploration

Before training, I want to confirm the generated YOLO dataset is complete and that the split looks reasonable.

In [ ]:
for split in ['train', 'valid', 'test']:
    assert (DATASET_DIR / split / 'images').exists(), f'Missing {split}/images'
    assert (DATASET_DIR / split / 'labels').exists(), f'Missing {split}/labels'

assert DATA_YAML.exists(), 'Missing data.yaml'
print('Dataset folders are ready.')

In [ ]:
with DATA_YAML.open('r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

data_config

In [ ]:
class_names = data_config['names']
id_to_class = {idx: name for idx, name in enumerate(class_names)}
class_to_id = {name: idx for idx, name in id_to_class.items()}

class_names

In [ ]:
def image_paths_for_split(split):
    image_dir = DATASET_DIR / split / 'images'
    return sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)


def label_paths_for_split(split):
    label_dir = DATASET_DIR / split / 'labels'
    return sorted(label_dir.glob('*.txt'))


def label_path_for_image(split, image_path):
    return DATASET_DIR / split / 'labels' / f'{image_path.stem}.txt'


split_images = {split: image_paths_for_split(split) for split in ['train', 'valid', 'test']}
split_labels = {split: label_paths_for_split(split) for split in ['train', 'valid', 'test']}

summary_df = pd.DataFrame({
    'split': ['train', 'valid', 'test'],
    'images': [len(split_images[split]) for split in ['train', 'valid', 'test']],
    'labels': [len(split_labels[split]) for split in ['train', 'valid', 'test']],
})

summary_df

In [ ]:
random.seed(42)
all_image_records = [(split, path) for split, paths in split_images.items() for path in paths]
preview_records = random.sample(all_image_records, k=min(6, len(all_image_records)))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for ax, (split, image_path) in zip(axes, preview_records):
    image = Image.open(image_path).convert('RGB')
    ax.imshow(image)
    ax.set_title(split)
    ax.axis('off')

for ax in axes[len(preview_records):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def read_yolo_label(label_path):
    rows = []
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        try:
            rows.append({
                'class_id': int(parts[0]),
                'x_center': float(parts[1]),
                'y_center': float(parts[2]),
                'width': float(parts[3]),
                'height': float(parts[4]),
            })
        except ValueError:
            continue
    return rows


sample_split, sample_image = preview_records[0]
sample_label = label_path_for_image(sample_split, sample_image)
pd.DataFrame(read_yolo_label(sample_label)).head()

In [ ]:
class_distribution = pd.DataFrame(0, index=['train', 'valid', 'test'], columns=class_names)

for split, label_paths in split_labels.items():
    counts = Counter()
    for label_path in label_paths:
        for row in read_yolo_label(label_path):
            if row['class_id'] in id_to_class:
                counts[id_to_class[row['class_id']]] += 1
    for class_name, count in counts.items():
        class_distribution.loc[split, class_name] = count

class_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(summary_df['split'], summary_df['images'])
ax.set_title('Images per Split')
ax.set_xlabel('Split')
ax.set_ylabel('Image Count')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
class_distribution.plot(kind='bar', ax=ax)
ax.set_title('Annotation Counts by Class')
ax.set_xlabel('Split')
ax.set_ylabel('Annotation Count')
ax.legend(title='Class')
plt.tight_layout()
plt.show()

## 5. Data Validation

These checks are intentionally lightweight. They catch the common dataset problems that can quietly break a YOLO training run.

In [ ]:
label_format_issues = []

for split, label_paths in split_labels.items():
    for label_path in label_paths:
        for line_number, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), start=1):
            parts = line.strip().split()
            if len(parts) != 5:
                label_format_issues.append((split, label_path.name, line_number, 'wrong field count'))
                continue
            try:
                int(parts[0])
                [float(value) for value in parts[1:]]
            except ValueError:
                label_format_issues.append((split, label_path.name, line_number, 'non-numeric value'))

print(f'Label format issues: {len(label_format_issues)}')
label_format_issues[:5]

In [ ]:
corrupted_images = []

for split, image_paths in split_images.items():
    for image_path in image_paths:
        try:
            with Image.open(image_path) as image:
                image.verify()
        except (UnidentifiedImageError, OSError) as error:
            corrupted_images.append({'split': split, 'path': str(image_path), 'error': str(error)})

print(f'Corrupted images: {len(corrupted_images)}')

In [ ]:
missing_labels = []
missing_images = []

for split in ['train', 'valid', 'test']:
    image_stems = {path.stem for path in split_images[split]}
    label_stems = {path.stem for path in split_labels[split]}
    missing_labels.extend((split, stem) for stem in sorted(image_stems - label_stems))
    missing_images.extend((split, stem) for stem in sorted(label_stems - image_stems))

print(f'Missing labels: {len(missing_labels)}')
print(f'Missing images: {len(missing_images)}')

In [ ]:
invalid_class_rows = []
valid_class_ids = set(id_to_class.keys())

for split, label_paths in split_labels.items():
    for label_path in label_paths:
        for line_number, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), start=1):
            parts = line.strip().split()
            if not parts:
                continue
            try:
                class_id = int(parts[0])
            except ValueError:
                invalid_class_rows.append((split, label_path.name, line_number, parts[0]))
                continue
            if class_id not in valid_class_ids:
                invalid_class_rows.append((split, label_path.name, line_number, class_id))

print(f'Invalid class IDs: {len(invalid_class_rows)}')

In [ ]:
validation_summary = {
    'corrupted_images': len(corrupted_images),
    'missing_labels': len(missing_labels),
    'missing_images': len(missing_images),
    'invalid_class_ids': len(invalid_class_rows),
    'label_format_issues': len(label_format_issues),
}

validation_summary

In [ ]:
assert all(count == 0 for count in validation_summary.values()), validation_summary
print('Validation passed.')

## 6. Annotation Visualization

The counts are useful, but object detection data also needs visual inspection. I want to see whether boxes land on the visible symptom area and whether labels match the basil condition.

In [ ]:
random.seed(7)
annotation_examples = random.sample(all_image_records, k=min(6, len(all_image_records)))
len(annotation_examples)

In [ ]:
def yolo_to_pixels(row, image_width, image_height):
    x_center = row['x_center'] * image_width
    y_center = row['y_center'] * image_height
    box_width = row['width'] * image_width
    box_height = row['height'] * image_height
    left = max(0, x_center - box_width / 2)
    top = max(0, y_center - box_height / 2)
    right = min(image_width, x_center + box_width / 2)
    bottom = min(image_height, y_center + box_height / 2)
    return left, top, right, bottom


def draw_annotations(image_path, label_path):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    width, height = image.size

    for row in read_yolo_label(label_path):
        if row['class_id'] not in id_to_class:
            continue
        class_name = id_to_class[row['class_id']]
        left, top, right, bottom = yolo_to_pixels(row, width, height)
        draw.rectangle((left, top, right, bottom), outline='red', width=max(2, width // 250))
        draw.text((left + 3, max(0, top - 12)), class_name, fill='red', font=font)

    return image

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

for ax, (split, image_path) in zip(axes, annotation_examples):
    label_path = label_path_for_image(split, image_path)
    annotated = draw_annotations(image_path, label_path)
    ax.imshow(annotated)
    ax.set_title(split)
    ax.axis('off')

for ax in axes[len(annotation_examples):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

While reviewing these examples, I am checking that boxes cover the relevant basil leaf area, class names match the visual condition, and the dataset does not include unrelated plants.

## 7. Training Setup

The first baseline uses `yolov8n.pt`. The nano model trains quickly and gives a practical starting point before trying larger models.

In [ ]:
model = YOLO('yolov8n.pt')

In [ ]:
EPOCHS = 80
IMAGE_SIZE = 640
BATCH_SIZE = 8
PATIENCE = 15
OPTIMIZER = 'AdamW'
SEED = 42

In [ ]:
training_config = {
    'model': 'yolov8n.pt',
    'data': str(DATA_YAML),
    'epochs': EPOCHS,
    'imgsz': IMAGE_SIZE,
    'batch': BATCH_SIZE,
    'patience': PATIENCE,
    'optimizer': OPTIMIZER,
    'seed': SEED,
    'project': YOLO_PROJECT,
    'name': RUN_NAME,
}

pd.DataFrame([training_config]).T.rename(columns={0: 'value'})

## 8. Training

YOLO reads the processed dataset from `data.yaml`, trains on the train split, and watches the valid split for early stopping. Run this cell only in Colab with a GPU runtime.

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    optimizer=OPTIMIZER,
    seed=SEED,
    project=YOLO_PROJECT,
    name=RUN_NAME,
    exist_ok=True,
)

## 9. Evaluation

After training, I reload the best checkpoint instead of the last checkpoint. This keeps evaluation aligned with the model that performed best on validation during training.

In [ ]:
RUN_DIR = Path(YOLO_PROJECT) / RUN_NAME
BEST_MODEL_PATH = RUN_DIR / 'weights' / 'best.pt'

best_model = YOLO(str(BEST_MODEL_PATH))
BEST_MODEL_PATH

In [ ]:
test_metrics = best_model.val(data=str(DATA_YAML), split='test')

In [ ]:
print(f'mAP50: {test_metrics.box.map50:.4f}')
print(f'mAP50-95: {test_metrics.box.map:.4f}')
print(f'Precision: {test_metrics.box.mp:.4f}')
print(f'Recall: {test_metrics.box.mr:.4f}')

`mAP50` measures detection quality at an IoU threshold of 0.50. For Rayhana, recall matters because missing a disease symptom can be more harmful than showing an extra low-confidence warning.

In [ ]:
VAL_DIR = Path(test_metrics.save_dir)
confusion_matrix_path = VAL_DIR / 'confusion_matrix.png'

if confusion_matrix_path.exists():
    image = Image.open(confusion_matrix_path)
    plt.figure(figsize=(7, 7))
    plt.imshow(image)
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.show()
else:
    print('Confusion matrix was not found in the validation output folder.')

In [ ]:
curve_files = ['results.png', 'P_curve.png', 'R_curve.png', 'PR_curve.png', 'F1_curve.png']

for file_name in curve_files:
    curve_path = RUN_DIR / file_name
    if curve_path.exists():
        image = Image.open(curve_path)
        plt.figure(figsize=(8, 5))
        plt.imshow(image)
        plt.axis('off')
        plt.title(file_name)
        plt.show()

## 10. Prediction Samples

The final check is visual: run the trained model on random test images and inspect the predicted boxes and confidence scores.

In [ ]:
random.seed(42)
test_images = split_images['test']
prediction_images = random.sample(test_images, k=min(6, len(test_images)))

prediction_images[:3]

In [ ]:
prediction_results = best_model.predict(
    source=[str(path) for path in prediction_images],
    conf=0.25,
    save=True,
    project=str(FIGURES_DIR),
    name='basil_prediction_samples',
    exist_ok=True,
)

In [ ]:
PREDICTION_SAVE_DIR = Path(prediction_results[0].save_dir)
saved_predictions = sorted(path for path in PREDICTION_SAVE_DIR.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

for ax, image_path in zip(axes, saved_predictions[:6]):
    image = Image.open(image_path).convert('RGB')
    ax.imshow(image)
    ax.set_title(image_path.name[:35])
    ax.axis('off')

for ax in axes[len(saved_predictions[:6]):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
confidence_rows = []

for result in prediction_results:
    image_name = Path(result.path).name
    for box in result.boxes:
        class_id = int(box.cls.item())
        confidence_rows.append({
            'image': image_name,
            'class': result.names[class_id],
            'confidence': float(box.conf.item()),
        })

confidence_df = pd.DataFrame(confidence_rows)
confidence_df.sort_values('confidence', ascending=False).head(20)

When reviewing predictions, I am looking for practical Rayhana behavior: confident detections should appear on basil leaves, low-confidence detections should be treated carefully, and repeated confusion between `fusarium` and `powdery` suggests the dataset needs more examples or cleaner labels.

## 11. Saving Outputs

The trained model, metrics, and visual prediction samples are saved into ignored local output folders. They are useful artifacts, but they should not be committed to GitHub.

In [ ]:
FINAL_MODEL_DIR = PROJECT_DIR / 'models/yolo'
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_PATH = FINAL_MODEL_DIR / 'basil_yolov8n_best.pt'
shutil.copy2(BEST_MODEL_PATH, FINAL_MODEL_PATH)

FINAL_MODEL_PATH

In [ ]:
metrics_output = {
    'model': 'yolov8n',
    'classes': class_names,
    'map50': float(test_metrics.box.map50),
    'map50_95': float(test_metrics.box.map),
    'precision': float(test_metrics.box.mp),
    'recall': float(test_metrics.box.mr),
    'best_model_path': str(FINAL_MODEL_PATH),
}

METRICS_PATH = METRICS_DIR / 'basil_yolov8n_metrics.json'
METRICS_PATH.write_text(json.dumps(metrics_output, indent=2), encoding='utf-8')

METRICS_PATH

In [ ]:
FINAL_PREDICTION_DIR = FIGURES_DIR / 'basil_yolov8n_predictions'
FINAL_PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

for image_path in saved_predictions:
    shutil.copy2(image_path, FINAL_PREDICTION_DIR / image_path.name)

FINAL_PREDICTION_DIR

In [ ]:
print('Best model:', FINAL_MODEL_PATH)
print('Metrics:', METRICS_PATH)
print('Prediction samples:', FINAL_PREDICTION_DIR)
print('Training run:', RUN_DIR)

## 12. Final Notes

Current limitations:

- The basil detection dataset is small, so validation metrics can move between runs.
- Only three classes are available: `fusarium`, `healthy`, and `powdery`.
- The model should not be presented as a general plant diagnosis system. Rayhana is basil-only.
- Field images from real Rayhana users may look different from this dataset, so future testing should include phone-camera basil scans.

Rayhana integration path:

1. Export the best YOLO model.
2. Add a backend inference endpoint for basil scans.
3. Return detected class, confidence, and bounding boxes to the app.
4. Use the detection result to trigger basil-specific care guidance.